# Araç Paylaşım Sistemi — Colab İnceleme Defteri

Bu notebook, **Araç Paylaşım Sistemi (Car Share)** kodlarını incelemek için hazırlanmıştır. Proje iki ana dosyadan oluşmaktadır:
1. `arac_backend.py` (Backend sınıfları ve iş mantığı)
2. `arac_gui.py` (Frontend, PyQt5 masaüstü arayüzü)

## Önemli Not
PyQt5 Google Colab üzerinde görsel bir pencere açamaz ancak SQLite veritabanı yönetimi ve OOP sınıf yapıları sorunsuz çalıştırılabilir/incelenebilir.


## 1. Arka Plan (Backend) ve Veri Modelleri
Aşağıdaki kod bloğu `arac_backend.py` dosyasına aittir. 
Araçlar, Kullanıcılar ve Paylaşım Sistemi sınıflarını (OOP) içerir. Ayrıca harita üzerinden mesafe hesaplama benzeri `get_mesafe` gibi simüle edilmiş lojistik algoritmalar barındırır.


In [ ]:
import sqlite3
from datetime import datetime

MESAFELER = {
    ("İstanbul", "Ankara"): 450, ("İstanbul", "İzmir"): 480, ("İstanbul", "Bursa"): 150, ("İstanbul", "Antalya"): 700,
    ("Ankara", "İzmir"): 590, ("Ankara", "Bursa"): 390, ("Ankara", "Antalya"): 480,
    ("İzmir", "Bursa"): 350, ("İzmir", "Antalya"): 460, ("Bursa", "Antalya"): 540
}

def get_mesafe(s1, s2):
    if s1 == s2: return 50
    if (s1, s2) in MESAFELER: return MESAFELER[(s1, s2)]
    if (s2, s1) in MESAFELER: return MESAFELER[(s2, s1)]
    return 300

class Arac:
    def __init__(self, arac_id, marka, model, tip, yakit_tipi, sehir, kilometre, km_ucreti, musait_mi=True):
        self._arac_id       = arac_id
        self._marka         = marka
        self._model         = model
        self._tip           = tip
        self._yakit_tipi    = yakit_tipi
        self._sehir         = sehir
        self._kilometre     = kilometre
        self._musait_mi     = musait_mi
        self._km_ucreti     = km_ucreti

    def get_arac_id(self):       return self._arac_id
    def get_marka(self):         return self._marka
    def get_model(self):         return self._model
    def get_tip(self):           return self._tip
    def get_yakit_tipi(self):    return self._yakit_tipi
    def get_sehir(self):         return self._sehir
    def get_kilometre(self):     return self._kilometre
    def get_musait_mi(self):     return self._musait_mi
    def get_km_ucreti(self):     return self._km_ucreti
    def get_tam_ad(self):        return f"{self._marka} {self._model}"

    def arac_bilgisi_guncelle(self, marka, model, tip, yakit_tipi, sehir, kilometre, km_ucreti):
        self._marka = marka
        self._model = model
        self._tip = tip
        self._yakit_tipi = yakit_tipi
        self._sehir = sehir
        self._kilometre = kilometre
        self._km_ucreti = km_ucreti

    def arac_durumu_guncelle(self, musait):
        self._musait_mi = musait

    def kilometre_guncelle(self, eklenen_km):
        if eklenen_km < 0:
            raise ValueError("Km negatif olamaz.")
        self._kilometre += eklenen_km

class Kullanici:
    def __init__(self, kullanici_id, ad, ehliyet_no, telefon=""):
        self._kullanici_id = kullanici_id
        self._ad           = ad
        self._ehliyet_no   = ehliyet_no
        self._telefon      = telefon
        self._kiralamalar  = []

    def get_kullanici_id(self): return self._kullanici_id
    def get_ad(self):           return self._ad
    def get_ehliyet_no(self):   return self._ehliyet_no
    def get_telefon(self):      return self._telefon

    def kiralama_ekle(self, kiralama):
        self._kiralamalar.append(kiralama)

    def kiralama_gecmisi(self):
        return list(self._kiralamalar)

    def aktif_kiralama(self):
        for k in self._kiralamalar:
            if k.get_aktif():
                return k
        return None

class Kiralama:
    def __init__(self, kiralama_id, arac: Arac, kullanici: Kullanici, alis_sehri, birakis_sehri, baslangic_saati: datetime, bitis_saati=None, aktif=True, toplam_ucret=0.0):
        self._kiralama_id     = kiralama_id
        self._arac            = arac
        self._kullanici       = kullanici
        self._alis_sehri      = alis_sehri
        self._birakis_sehri   = birakis_sehri
        self._baslangic_saati = baslangic_saati
        self._bitis_saati     = bitis_saati
        self._aktif           = aktif
        self._toplam_ucret    = toplam_ucret

    def get_kiralama_id(self):     return self._kiralama_id
    def get_arac(self):            return self._arac
    def get_kullanici(self):       return self._kullanici
    def get_alis_sehri(self):      return self._alis_sehri
    def get_birakis_sehri(self):   return self._birakis_sehri
    def get_baslangic_saati(self): return self._baslangic_saati
    def get_bitis_saati(self):     return self._bitis_saati
    def get_aktif(self):           return self._aktif
    def get_toplam_ucret(self):    return self._toplam_ucret

    def kiralama_baslat(self):
        self._arac.arac_durumu_guncelle(False)
        return True, f"✓ Kiralama #{self._kiralama_id} başlatıldı."

    def kiralama_bitir(self, bitis_saati: datetime, eklenen_km: int = 0):
        if not self._aktif:
            return False, "Bu kiralama zaten sona ermiş."
        if bitis_saati <= self._baslangic_saati:
            return False, "Bitiş saati başlangıçtan önce olamaz."

        self._bitis_saati = bitis_saati
        self._aktif       = False
        sure_saat         = (bitis_saati - self._baslangic_saati).total_seconds() / 3600
        self._toplam_ucret = round(sure_saat * self._arac.get_saatlik_ucret(), 2)
        self._arac.kilometre_guncelle(eklenen_km)
        self._arac.arac_durumu_guncelle(True)
        return True, f"✓ Kiralama bitti. Süre: {sure_saat:.1f} saat | Ücret: ₺{self._toplam_ucret:.2f}"

    def kiralama_bilgisi(self):
        sure = "Devam ediyor"
        if self._bitis_saati:
            dk = int((self._bitis_saati - self._baslangic_saati).total_seconds() / 60)
            sure = f"{dk // 60}s {dk % 60}dk"
        return {
            "id":         self._kiralama_id,
            "arac":       self._arac.get_tam_ad(),
            "kullanici":  self._kullanici.get_ad(),
            "alis":       self._alis_sehri,
            "birakis":    self._birakis_sehri,
            "baslangic":  self._baslangic_saati.strftime("%d.%m.%Y %H:%M"),
            "bitis":      self._bitis_saati.strftime("%d.%m.%Y %H:%M") if self._bitis_saati else "—",
            "sure":       sure,
            "ucret":      f"₺{self._toplam_ucret:.2f}",
            "aktif":      self._aktif,
        }

class PaylasimSistemi:
    def __init__(self, db_path="carshare.db"):
        self.db_path = db_path
        self._init_db()

    def _get_conn(self):
        return sqlite3.connect(self.db_path)

    def _init_db(self):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute('''
            CREATE TABLE IF NOT EXISTS araclar (
                id INTEGER PRIMARY KEY,
                marka TEXT,
                model TEXT,
                tip TEXT,
                yakit_tipi TEXT,
                sehir TEXT,
                kilometre INTEGER,
                km_ucreti REAL,
                musait_mi INTEGER
            )
        ''')
        cur.execute('''
            CREATE TABLE IF NOT EXISTS kullanicilar (
                id INTEGER PRIMARY KEY,
                ad TEXT,
                ehliyet_no TEXT,
                telefon TEXT
            )
        ''')
        cur.execute('''
            CREATE TABLE IF NOT EXISTS kiralamalar (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                arac_id INTEGER,
                kullanici_id INTEGER,
                alis_sehri TEXT,
                birakis_sehri TEXT,
                baslangic TEXT,
                bitis TEXT,
                aktif INTEGER,
                ucret REAL,
                eklenen_km INTEGER DEFAULT 0
            )
        ''')
        conn.commit()

        cur.execute("SELECT COUNT(*) FROM araclar")
        if cur.fetchone()[0] == 0:
            self._dummy_veri_olustur(conn)
            
        # Eğer hiç aktif kiralama yoksa, müsait araçlar ve kişiler ile 5 adet aktif kiralama oluştur
        cur.execute("SELECT COUNT(*) FROM kiralamalar WHERE aktif=1")
        if cur.fetchone()[0] == 0:
            cur.execute('SELECT id FROM araclar WHERE musait_mi=1 LIMIT 5')
            cars = [r[0] for r in cur.fetchall()]
            cur.execute('SELECT id FROM kullanicilar WHERE id NOT IN (SELECT kullanici_id FROM kiralamalar WHERE aktif=1) LIMIT 5')
            users = [r[0] for r in cur.fetchall()]
            
            from datetime import datetime, timedelta
            for i in range(min(len(cars), len(users))):
                car_id, user_id = cars[i], users[i]
                bas = datetime.now() - timedelta(hours=(i+1)*6, minutes=i*17)
                cur.execute('SELECT km_ucreti FROM araclar WHERE id=?', (car_id,))
                km_ucreti = cur.fetchone()[0]
                ucret = 50 * km_ucreti
                cur.execute('''
                    INSERT INTO kiralamalar (arac_id, kullanici_id, alis_sehri, birakis_sehri, baslangic, bitis, aktif, ucret)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                ''', (car_id, user_id, 'İstanbul', 'İstanbul', bas.strftime('%Y-%m-%d %H:%M:%S'), None, 1, ucret))
                cur.execute('UPDATE araclar SET musait_mi=0 WHERE id=?', (car_id,))
            conn.commit()

        conn.close()

    def _dummy_veri_olustur(self, conn):
        cur = conn.cursor()
        araclar = [
            (1, "Toyota", "Corolla", "Sedan", "Benzin", "İstanbul", 45000, 3.5, 1),
            (2, "Ford", "Focus", "Hatchback", "Dizel", "Ankara", 62000, 3.0, 1),
            (3, "BMW", "X3", "SUV", "Dizel", "İzmir", 28000, 5.5, 1),
            (4, "Tesla", "Model 3", "Elektrikli", "Elektrik", "İstanbul", 8000, 4.0, 1),
            (5, "Renault", "Clio", "Hatchback", "Benzin", "Bursa", 91000, 2.5, 1),
            (6, "Mercedes", "Vito", "Minivan", "Dizel", "Antalya", 55000, 6.0, 1),
            (7, "Hyundai", "i20", "Hatchback", "Benzin", "İzmir", 30000, 2.8, 1),
            (8, "Volkswagen", "Golf", "Hatchback", "Benzin", "İstanbul", 42000, 3.2, 1),
            (9, "Honda", "Civic", "Sedan", "Benzin", "Ankara", 21000, 3.5, 1),
            (10, "Audi", "A4", "Sedan", "Dizel", "Bursa", 15000, 5.0, 1),
            (11, "Kia", "Sportage", "SUV", "Dizel", "Antalya", 60000, 4.2, 1),
            (12, "Peugeot", "3008", "SUV", "Dizel", "İstanbul", 48000, 4.5, 1),
            (13, "Toyota", "Yaris", "Hatchback", "Hibrit", "İzmir", 12000, 2.6, 1),
            (14, "Volvo", "XC90", "SUV", "Dizel", "Ankara", 19000, 7.0, 1),
            (15, "Porsche", "Macan", "SUV", "Benzin", "İstanbul", 5000, 9.0, 1),
            (16, "Fiat", "Egea", "Sedan", "Dizel", "Bursa", 15000, 2.5, 1),
            (17, "Renault", "Megane", "Sedan", "Dizel", "İstanbul", 32000, 3.0, 1),
            (18, "Dacia", "Duster", "SUV", "Dizel", "Antalya", 45000, 3.2, 1),
            (19, "Hyundai", "Tucson", "SUV", "Benzin", "İzmir", 22000, 4.0, 1),
            (20, "Nissan", "Qashqai", "SUV", "Benzin", "Ankara", 37000, 4.1, 1),
            (21, "Skoda", "Octavia", "Sedan", "Benzin", "İstanbul", 18000, 3.6, 1),
            (22, "Seat", "Leon", "Hatchback", "Benzin", "Bursa", 29000, 3.4, 1),
            (23, "Volkswagen", "Passat", "Sedan", "Dizel", "Ankara", 54000, 4.5, 1),
            (24, "Opel", "Astra", "Hatchback", "Benzin", "İzmir", 26000, 3.3, 1),
            (25, "Citroen", "C3", "Hatchback", "Dizel", "Antalya", 12000, 2.7, 1),
            (26, "Honda", "CR-V", "SUV", "Hibrit", "İstanbul", 41000, 4.8, 1),
            (27, "BMW", "520i", "Sedan", "Benzin", "Ankara", 15000, 6.5, 1),
            (28, "Mercedes", "C200", "Sedan", "Benzin", "İzmir", 11000, 6.2, 1),
            (29, "Audi", "Q5", "SUV", "Dizel", "Bursa", 33000, 5.8, 1),
            (30, "Kia", "Ceed", "Hatchback", "Dizel", "İstanbul", 21000, 3.1, 1),
            (31, "Ford", "Kuga", "SUV", "Dizel", "Antalya", 47000, 4.3, 1),
            (32, "Tesla", "Model Y", "Elektrikli", "Elektrik", "İstanbul", 5000, 4.5, 1),
            (33, "TOGG", "T10X", "SUV", "Elektrik", "Bursa", 2000, 3.8, 1),
            (34, "MG", "ZS EV", "SUV", "Elektrik", "İzmir", 8000, 3.5, 1),
            (35, "Hyundai", "Ioniq 5", "SUV", "Elektrik", "Ankara", 10000, 4.0, 1),
        ]
        cur.executemany("INSERT INTO araclar VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", araclar)

        kullanicilar = [
            (1, "Ahmet Yılmaz", "12ABC123", "0532 111 22 33"),
            (2, "Ayşe Kaya", "34DEF456", "0541 333 44 55"),
            (3, "Mehmet Demir", "56GHI789", "0555 666 77 88"),
            (4, "Zeynep Çelik", "78JKL012", "0544 999 00 11"),
            (5, "Caner Türk", "90MNO345", "0533 222 33 44"),
            (6, "Elif Can", "11PQR678", "0531 444 55 66"),
            (7, "Emre Şahin", "22STU901", "0530 555 66 77"),
            (8, "Fatma Yıldız", "33VWX234", "0543 666 77 88"),
            (9, "Kemal Kara", "44YZA567", "0542 777 88 99"),
            (10, "Leyla Ak", "55BCD890", "0546 888 99 00"),
            (11, "Murat Boz", "66EFG123", "0532 999 00 11"),
            (12, "Neslihan Yurt", "77HIJ456", "0541 000 11 22"),
            (13, "Orhan Veli", "88KLM789", "0555 111 22 33"),
            (14, "Pelin Su", "99NOP012", "0544 222 33 44"),
            (15, "Rıza Sarraf", "00QRS345", "0533 333 44 55")
        ]
        cur.executemany("INSERT INTO kullanicilar VALUES (?, ?, ?, ?)", kullanicilar)
        
        from datetime import timedelta
        for i in range(1, 11):
            bas = datetime.now() - timedelta(days=i, hours=2)
            bit = bas + timedelta(hours=3)
            ucret = 150 * araclar[i-1][7]
            cur.execute("""
                INSERT INTO kiralamalar (arac_id, kullanici_id, alis_sehri, birakis_sehri, baslangic, bitis, aktif, ucret)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (i, i, "İstanbul", "Ankara", bas.strftime("%Y-%m-%d %H:%M:%S"), bit.strftime("%Y-%m-%d %H:%M:%S"), 0, ucret))
        conn.commit()

    def _get_arac_from_row(self, row):
        return Arac(row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], bool(row[8]))

    def _get_kullanici_from_row(self, row):
        return Kullanici(row[0], row[1], row[2], row[3])

    def get_araclar(self):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT * FROM araclar")
        araclar = {}
        for r in cur.fetchall():
            araclar[r[0]] = self._get_arac_from_row(r)
        conn.close()
        return araclar

    def get_musait_araclar(self):
        return {k: v for k, v in self.get_araclar().items() if v.get_musait_mi()}

    def arac_ekle(self, arac: Arac):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id FROM araclar WHERE id=?", (arac.get_arac_id(),))
        if cur.fetchone():
            conn.close()
            return False, f"ID {arac.get_arac_id()} zaten kayıtlı."
        cur.execute("INSERT INTO araclar VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", (
            arac.get_arac_id(), arac.get_marka(), arac.get_model(), arac.get_tip(), arac.get_yakit_tipi(), arac.get_sehir(),
            arac.get_kilometre(), arac.get_km_ucreti(), 1 if arac.get_musait_mi() else 0
        ))
        conn.commit()
        conn.close()
        return True, f"✓ '{arac.get_tam_ad()}' sisteme eklendi."

    def arac_guncelle(self, arac_id, marka, model, tip, yakit_tipi, sehir, kilometre, km_ucreti):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id FROM araclar WHERE id=?", (arac_id,))
        if not cur.fetchone():
            conn.close()
            return False, "Araç bulunamadı."
        cur.execute("UPDATE araclar SET marka=?, model=?, tip=?, yakit_tipi=?, sehir=?, kilometre=?, km_ucreti=? WHERE id=?", (
            marka, model, tip, yakit_tipi, sehir, kilometre, km_ucreti, arac_id
        ))
        conn.commit()
        conn.close()
        return True, "✓ Araç bilgileri güncellendi."

    def arac_sil(self, arac_id):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT musait_mi FROM araclar WHERE id=?", (arac_id,))
        row = cur.fetchone()
        if not row:
            conn.close()
            return False, "Araç bulunamadı."
        if not row[0]:
            conn.close()
            return False, "Araç şu an kirada olduğu için silinemez."
        cur.execute("DELETE FROM araclar WHERE id=?", (arac_id,))
        conn.commit()
        conn.close()
        return True, "✓ Araç başarıyla silindi."

    def get_kullanicilar(self):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT * FROM kullanicilar")
        kullanicilar = {}
        for r in cur.fetchall():
            u = self._get_kullanici_from_row(r)
            cur.execute("SELECT * FROM kiralamalar WHERE kullanici_id=?", (u.get_kullanici_id(),))
            for kr in cur.fetchall():
                cur.execute("SELECT * FROM araclar WHERE id=?", (kr[1],))
                arow = cur.fetchone()
                if arow:
                    a = self._get_arac_from_row(arow)
                    bas = datetime.strptime(kr[5], "%Y-%m-%d %H:%M:%S")
                    bit = datetime.strptime(kr[6], "%Y-%m-%d %H:%M:%S") if kr[6] else None
                    k = Kiralama(kr[0], a, u, kr[3], kr[4], bas, bit, bool(kr[7]), kr[8])
                    u.kiralama_ekle(k)
            kullanicilar[r[0]] = u
        conn.close()
        return kullanicilar

    def kullanici_ekle(self, kullanici: Kullanici):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT id FROM kullanicilar WHERE id=?", (kullanici.get_kullanici_id(),))
        if cur.fetchone():
            conn.close()
            return False, f"ID {kullanici.get_kullanici_id()} zaten kayıtlı."
        cur.execute("SELECT id FROM kullanicilar WHERE ehliyet_no=?", (kullanici.get_ehliyet_no(),))
        if cur.fetchone():
            conn.close()
            return False, "Bu ehliyet numarası zaten kayıtlı."
        cur.execute("INSERT INTO kullanicilar VALUES (?, ?, ?, ?)", (
            kullanici.get_kullanici_id(), kullanici.get_ad(), kullanici.get_ehliyet_no(), kullanici.get_telefon()
        ))
        conn.commit()
        conn.close()
        return True, f"✓ '{kullanici.get_ad()}' kayıt edildi."

    def kiralama_baslat(self, arac_id, kullanici_id, alis_sehri, birakis_sehri, baslangic: datetime):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT musait_mi, km_ucreti FROM araclar WHERE id=?", (arac_id,))
        arow = cur.fetchone()
        if not arow:
            conn.close(); return False, "Araç bulunamadı."
        if not arow[0]:
            conn.close(); return False, "Araç şu an başka bir kullanıcıda."
        
        km_ucreti = arow[1]
        
        cur.execute("SELECT id FROM kullanicilar WHERE id=?", (kullanici_id,))
        if not cur.fetchone():
            conn.close(); return False, "Kullanıcı bulunamadı."

        cur.execute("SELECT id FROM kiralamalar WHERE kullanici_id=? AND aktif=1", (kullanici_id,))
        if cur.fetchone():
            conn.close(); return False, "Kullanıcı zaten aktif bir kiralamaya sahip."
            
        mesafe = get_mesafe(alis_sehri, birakis_sehri)
        toplam_ucret = round(mesafe * km_ucreti, 2)

        cur.execute("UPDATE araclar SET musait_mi=0 WHERE id=?", (arac_id,))
        cur.execute('''
            INSERT INTO kiralamalar (arac_id, kullanici_id, alis_sehri, birakis_sehri, baslangic, aktif, ucret)
            VALUES (?, ?, ?, ?, ?, 1, ?)
        ''', (arac_id, kullanici_id, alis_sehri, birakis_sehri, baslangic.strftime("%Y-%m-%d %H:%M:%S"), toplam_ucret))
        conn.commit()
        conn.close()
        return True, "✓ Kiralama başlatıldı."

    def kiralama_bitir(self, kiralama_id, bitis: datetime):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT arac_id, alis_sehri, birakis_sehri, baslangic, aktif FROM kiralamalar WHERE id=?", (kiralama_id,))
        row = cur.fetchone()
        if not row:
            conn.close(); return False, "Kiralama bulunamadı."
        arac_id, alis_sehri, birakis_sehri, bas_str, aktif = row
        if not aktif:
            conn.close(); return False, "Bu kiralama zaten sona ermiş."
        
        baslangic = datetime.strptime(bas_str, "%Y-%m-%d %H:%M:%S")
        if bitis <= baslangic:
            conn.close(); return False, "Bitiş saati başlangıçtan önce olamaz."

        cur.execute("SELECT km_ucreti FROM araclar WHERE id=?", (arac_id,))
        km_ucreti = cur.fetchone()[0]

        mesafe = get_mesafe(alis_sehri, birakis_sehri)
        toplam_ucret = round(mesafe * km_ucreti, 2)

        cur.execute("UPDATE kiralamalar SET bitis=?, aktif=0, ucret=? WHERE id=?", 
            (bitis.strftime("%Y-%m-%d %H:%M:%S"), toplam_ucret, kiralama_id))
        cur.execute("UPDATE araclar SET musait_mi=1, kilometre=kilometre+? WHERE id=?", (mesafe, arac_id))
        conn.commit()
        conn.close()
        return True, f"✓ Kiralama bitti. Mesafe: {mesafe} km | Ücret: ₺{toplam_ucret:.2f}"

    def get_kiralamalar(self):
        araclar = self.get_araclar()
        kullanicilar = self.get_kullanicilar()
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT * FROM kiralamalar ORDER BY id ASC")
        kiralamalar = []
        for kr in cur.fetchall():
            a = araclar.get(kr[1])
            u = kullanicilar.get(kr[2])
            if not a or not u: continue
            bas = datetime.strptime(kr[5], "%Y-%m-%d %H:%M:%S")
            bit = datetime.strptime(kr[6], "%Y-%m-%d %H:%M:%S") if kr[6] else None
            k = Kiralama(kr[0], a, u, kr[3], kr[4], bas, bit, bool(kr[7]), kr[8])
            kiralamalar.append(k)
        conn.close()
        return kiralamalar

    def get_aktif_kiralamalar(self):
        return [k for k in self.get_kiralamalar() if k.get_aktif()]

    def get_gecmis_kiralamalar(self):
        return [k for k in self.get_kiralamalar() if not k.get_aktif()]

    def toplam_gelir(self):
        conn = self._get_conn()
        cur = conn.cursor()
        cur.execute("SELECT SUM(ucret) FROM kiralamalar WHERE aktif=0")
        val = cur.fetchone()[0]
        conn.close()
        return val if val else 0.0

## 2. Kullanıcı Arayüzü (GUI) ve Sayfa Geçişleri
Aşağıdaki kodlar `arac_gui.py` dosyasının parçalarıdır. 
Aracın kiralanması, harita konum seçimi ve kullanıcı paneli gibi sayfaları barındıran PyQt5 arayüz kodlarıdır. Uzun olduğu için parçalar halinde eklenmiştir.


In [ ]:
import sys
from datetime import datetime, timedelta

from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QLabel, QPushButton, QTableWidget, QTableWidgetItem, QFrame,
    QStackedWidget, QHeaderView, QLineEdit, QGridLayout,
    QDateTimeEdit, QSpinBox, QDoubleSpinBox, QMessageBox,
    QAbstractItemView, QGraphicsDropShadowEffect, QComboBox, QScrollArea
)
from PyQt5.QtCore import Qt, QDateTime, QTimer, QRegExp
from PyQt5.QtGui import QColor, QPalette, QRegExpValidator

# ====== IMPORT FROM BACKEND ======
from arac_backend import Arac, Kullanici, PaylasimSistemi, get_mesafe
# =================================

# ══════════════════════════════════════════════════════════
#  RENK PALETİ
# ══════════════════════════════════════════════════════════
C = {
    "bg":        "#F4F7FB",
    "sidebar":   "#FFFFFF",
    "card":      "#FFFFFF",
    "border":    "#D6E4F0",
    "accent":    "#1565C0",
    "accent2":   "#0D47A1",
    "accent3":   "#42A5F5",
    "gold":      "#F57C00",
    "gold2":     "#FFB300",
    "success":   "#2ECC71",
    "warning":   "#F39C12",
    "danger":    "#E74C3C",
    "text":      "#1A2A3A",
    "text_sub":  "#4A6080",
    "text_dim":  "#90A8C0",
    "row_alt":   "#EEF4FB",
    "row_sel":   "#DBEAFE",
    "input_bg":  "#F4F7FB",
    "tag_bg":    "#EBF4FF",
    "sidebar_w": 240,
}

ARAC_TIPLERI = ["Sedan", "SUV", "Hatchback", "Minivan", "Elektrikli", "Kamyonet"]
YAKIT_TIPLERI = ["Benzin", "Dizel", "Elektrik", "Hibrit"]
SEHIRLER = ["İstanbul", "Ankara", "İzmir", "Bursa", "Antalya"]
MARKALAR     = ["Toyota", "Ford", "BMW", "Mercedes", "Tesla", "Renault", "Volkswagen", "Hyundai"]

# ── Stil yardımcıları ─────────────────────────────────────
def card_ss(border_color=None):
    bc = border_color or C['border']
    return f"""
        QFrame{{
            background:{C['card']};
            border:1.5px solid {bc};
            border-radius:16px;
        }}
    """

def btn_primary_ss():
    return f"""
        QPushButton{{
            background:qlineargradient(x1:0,y1:0,x2:1,y2:0, stop:0 {C['accent2']},stop:1 {C['accent']});
            color:#fff;border:none;border-radius:10px; padding:10px 22px;font-size:13px;font-weight:700;
        }}
        QPushButton:hover{{
            background:qlineargradient(x1:0,y1:0,x2:1,y2:0, stop:0 {C['accent']},stop:1 {C['accent3']});
        }}
        QPushButton:pressed{{background:{C['accent2']};}}
    """

def btn_gold_ss():
    return f"""
        QPushButton{{
            background:qlineargradient(x1:0,y1:0,x2:1,y2:0, stop:0 {C['gold']},stop:1 {C['gold2']});
            color:#fff;border:none;border-radius:10px; padding:10px 22px;font-size:13px;font-weight:700;
        }}
        QPushButton:hover{{
            background:qlineargradient(x1:0,y1:0,x2:1,y2:0, stop:0 {C['gold2']},stop:1 #FFD080);
        }}
    """

def btn_success_ss():
    return f"""
        QPushButton{{background:{C['success']};color:#fff;border:none;
        border-radius:10px;padding:10px 22px;font-size:13px;font-weight:700;}}
        QPushButton:hover{{background:#27AE60;}}
    """

def btn_danger_ss():
    return f"""
        QPushButton{{background:{C['danger']};color:#fff;border:none;
        border-radius:10px;padding:10px 22px;font-size:13px;font-weight:700;}}
        QPushButton:hover{{background:#C0392B;}}
    """

def btn_outline_ss():
    return f"""
        QPushButton{{
            background:transparent;color:{C['accent']};
            border:1.5px solid {C['accent']};border-radius:10px;
            padding:9px 20px;font-size:12px;font-weight:600;
        }}
        QPushButton:hover{{background:{C['tag_bg']};}}
    """

def input_ss():
    return f"""
        QLineEdit,QSpinBox,QDoubleSpinBox,QDateTimeEdit,QComboBox{{
            background:{C['input_bg']};color:{C['text']};
            border:1.5px solid {C['border']};border-radius:9px;
            padding:8px 12px;font-size:13px;
        }}
        QLineEdit:focus,QSpinBox:focus,QDoubleSpinBox:focus,
        QDateTimeEdit:focus,QComboBox:focus{{ border:1.5px solid {C['accent']}; }}
        QSpinBox::up-button,QSpinBox::down-button,
        QDoubleSpinBox::up-button,QDoubleSpinBox::down-button{{ background:{C['border']};width:18px;border-radius:3px; }}
        QDateTimeEdit::drop-down,QComboBox::drop-down{{ background:{C['border']};width:22px;border-radius:3px; }}
        QComboBox QAbstractItemView{{
            background:{C['card']};color:{C['text']};
            selection-background-color:{C['row_sel']};
            border:1px solid {C['border']};
        }}
    """

TABLE_SS = f"""
    QTableWidget{{
        background:{C['card']};color:{C['text']};border:none;
        gridline-color:{C['border']};font-size:13px;
        selection-background-color:{C['row_sel']};outline:none;
        alternate-background-color:{C['row_alt']};
    }}
    QTableWidget::item{{padding:9px 13px;border-bottom:1px solid {C['border']};}}
    QTableWidget::item:selected{{background:{C['row_sel']};color:{C['accent']};}}
    QHeaderView::section{{
        background:{C['bg']};color:{C['accent2']};
        padding:9px 13px;border:none;
        border-bottom:2px solid {C['accent']};
        font-size:11px;font-weight:700;letter-spacing:0.8px;
    }}
    QScrollBar:vertical{{background:{C['bg']};width:7px;border-radius:4px;}}
    QScrollBar::handle:vertical{{background:{C['border']};border-radius:4px;min-height:20px;}}
    QScrollBar::handle:vertical:hover{{background:{C['accent']};}}
    QScrollBar:horizontal{{background:{C['bg']};height:7px;border-radius:4px;}}
    QScrollBar::handle:horizontal{{background:{C['border']};border-radius:4px;}}
"""

def shadow(widget, blur=20, dy=4, alpha=25):
    fx = QGraphicsDropShadowEffect(widget)
    fx.setBlurRadius(blur)
    color = QColor(C['accent2'])
    color.setAlpha(alpha)
    fx.setColor(color)
    fx.setOffset(0, dy)
    widget.setGraphicsEffect(fx)


# ══════════════════════════════════════════════════════════
#  BİLEŞEN: KPI Kartı
# ══════════════════════════════════════════════════════════
class KpiCard(QFrame):
    def __init__(self, ikon, baslik, deger, alt, renk, parent=None):
        super().__init__(parent)
        self.setFixedHeight(118)
        self.setStyleSheet(f"""
            QFrame{{
                background:{C['card']};
                border:1.5px solid {C['border']};
                border-radius:16px;
                border-top:4px solid {renk};
            }}
        """)
        shadow(self, blur=16, dy=3)
        lay = QVBoxLayout(self)
        lay.setContentsMargins(18, 14, 18, 12)
        lay.setSpacing(3)
        top_row = QHBoxLayout()
        lbl_i = QLabel(ikon)
        lbl_i.setStyleSheet("font-size:22px;background:transparent;border:none;")
        lbl_t = QLabel(baslik.upper())
        lbl_t.setStyleSheet(f"color:{C['text_sub']};font-size:10px;font-weight:700;letter-spacing:1px;background:transparent;border:none;")
        top_row.addWidget(lbl_i); top_row.addWidget(lbl_t); top_row.addStretch()
        lay.addLayout(top_row)
        lbl_v = QLabel(str(deger))
        lbl_v.setStyleSheet(f"color:{renk};font-size:27px;font-weight:800;background:transparent;border:none;")
        lbl_a = QLabel(alt)
        lbl_a.setStyleSheet(f"color:{C['text_dim']};font-size:11px;background:transparent;border:none;")
        lay.addWidget(lbl_v); lay.addWidget(lbl_a)

# ══════════════════════════════════════════════════════════
#  BİLEŞEN: Sidebar Butonu
# ══════════════════════════════════════════════════════════
class SideBtn(QPushButton):
    def __init__(self, ikon, metin, parent=None):
        super().__init__(f"  {ikon}  {metin}", parent)
        self.setCheckable(True)
        self.setFixedHeight(50)
        self.setCursor(Qt.PointingHandCursor)
        self._refresh(False)

    def _refresh(self, aktif):
        if aktif:
            self.setStyleSheet(f"""
                QPushButton{{
                    background:qlineargradient(x1:0,y1:0,x2:1,y2:0, stop:0 rgba(21,101,192,0.10), stop:1 rgba(21,101,192,0.03));
                    color:{C['accent']};
                    border:none;border-left:3px solid {C['accent']};
                    border-radius:0;text-align:left;padding-left:18px;
                    font-size:13px;font-weight:700;
                }}
            """)
        else:
            self.setStyleSheet(f"""
                QPushButton{{
                    background:transparent;color:{C['text_sub']};
                    border:none;border-left:3px solid transparent;
                    border-radius:0;text-align:left;padding-left:18px;font-size:13px;
                }}
                QPushButton:hover{{
                    background:rgba(21,101,192,0.05);color:{C['text']};
                    border-left:3px solid {C['accent3']};
                }}
            """)

    def setChecked(self, v):
        super().setChecked(v)
        self._refresh(v)

# ══════════════════════════════════════════════════════════
#  EKRAN 1: Dashboard
# ══════════════════════════════════════════════════════════
class DashboardPage(QWidget):
    def __init__(self, sistem: PaylasimSistemi, parent=None):
        super().__init__(parent)
        self.sistem = sistem
        self.setStyleSheet(f"background:{C['bg']};")
        self._build()

    def _build(self):
        lay = QVBoxLayout(self)
        lay.setContentsMargins(28, 22, 28, 22)
        lay.setSpacing(18)

        ust = QHBoxLayout()
        lbl = QLabel("🚗  Sistem Genel Bakışı")
        lbl.setStyleSheet(f"color:{C['text']};font-size:22px;font-weight:800;")
        ust.addWidget(lbl); ust.addStretch()
        self.lbl_tarih = QLabel()
        self.lbl_tarih.setStyleSheet(f"color:{C['text_sub']};font-size:12px;")
        ust.addWidget(self.lbl_tarih)
        lay.addLayout(ust)

        self.kpi_row = QHBoxLayout(); self.kpi_row.setSpacing(14)
        lay.addLayout(self.kpi_row)

        sub = QLabel("Aktif Kiralamalar")
        sub.setStyleSheet(f"color:{C['text_sub']};font-size:12px;font-weight:700;letter-spacing:0.5px;")
        lay.addWidget(sub)

        self.tablo = QTableWidget()
        self.tablo.setColumnCount(6)
        self.tablo.setHorizontalHeaderLabels(["Kiralama ID", "Araç", "Kullanıcı", "Başlangıç", "Saatlik Ücret", "Durum"])
        self.tablo.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.tablo.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.tablo.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.tablo.verticalHeader().setVisible(False)
        self.tablo.setAlternatingRowColors(True)
        self.tablo.setStyleSheet(TABLE_SS)
        lay.addWidget(self.tablo)
        self.refresh()

    def refresh(self):
        self.lbl_tarih.setText(datetime.now().strftime("%d %B %Y"))
        toplam_arac    = len(self.sistem.get_araclar())
        musait_arac    = len(self.sistem.get_musait_araclar())
        aktif_k        = len(self.sistem.get_aktif_kiralamalar())
        kullanici_say  = len(self.sistem.get_kullanicilar())
        gelir          = self.sistem.toplam_gelir()

        while self.kpi_row.count():
            item = self.kpi_row.takeAt(0)
            if item.widget(): item.widget().deleteLater()

        self.kpi_row.addWidget(KpiCard("🚗", "Toplam Araç",      toplam_arac,            "sistemde kayıtlı",      C['accent']))
        self.kpi_row.addWidget(KpiCard("✅", "Müsait Araç",      musait_arac,            "kiralama hazır",         C['success']))
        self.kpi_row.addWidget(KpiCard("🔑", "Aktif Kiralama",   aktif_k,                "devam eden kiralama",    C['warning']))
        self.kpi_row.addWidget(KpiCard("👤", "Kayıtlı Üye",      kullanici_say,          "sisteme kayıtlı",        C['gold']))
        self.kpi_row.addWidget(KpiCard("💰", "Toplam Gelir",      f"₺{gelir:,.0f}",      "tamamlanan kiralamar",   C['accent2']))

        self.tablo.setRowCount(0)
        for k in self.sistem.get_aktif_kiralamalar():
            bilgi = k.kiralama_bilgisi()
            r = self.tablo.rowCount(); self.tablo.insertRow(r)
            self.tablo.setItem(r, 0, QTableWidgetItem(str(bilgi["id"])))
            self.tablo.setItem(r, 1, QTableWidgetItem(bilgi["arac"]))
            self.tablo.setItem(r, 2, QTableWidgetItem(bilgi["kullanici"]))
            self.tablo.setItem(r, 3, QTableWidgetItem(bilgi["baslangic"]))
            self.tablo.setItem(r, 4, QTableWidgetItem(f"₺{k.get_arac().get_km_ucreti():.2f}/km"))
            durum = QTableWidgetItem("🔑  Devam Ediyor")
            durum.setForeground(QColor(C['warning']))
            self.tablo.setItem(r, 5, durum)
            self.tablo.setRowHeight(r, 44)

# ══════════════════════════════════════════════════════════
#  EKRAN 2: Araç Yönetimi
# ══════════════════════════════════════════════════════════
class AracPage(QWidget):
    def __init__(self, sistem: PaylasimSistemi, dashboard: DashboardPage, parent=None):
        super().__init__(parent)
        self.sistem    = sistem
        self.dashboard = dashboard
        self.setStyleSheet(f"background:{C['bg']};")
        self._build()

    def _lbl(self, txt):
        l = QLabel(txt)
        l.setStyleSheet(f"color:{C['text_sub']};font-size:11px;font-weight:700;letter-spacing:0.3px;")
        return l

    def _section_lbl(self, txt, renk):
        l = QLabel(txt)
        l.setStyleSheet(f"color:{renk};font-size:13px;font-weight:700;background:transparent;")
        return l

    def _build(self):
        scroll = QScrollArea(self)
        scroll.setWidgetResizable(True)
        scroll.setStyleSheet("QScrollArea{border:none;background:transparent;}")

        container = QWidget(); container.setStyleSheet(f"background:{C['bg']};")
        lay = QVBoxLayout(container)
        lay.setContentsMargins(28, 22, 28, 22)
        lay.setSpacing(16)

        lbl = QLabel("🚗  Araç Yönetimi")
        lbl.setStyleSheet(f"color:{C['text']};font-size:22px;font-weight:800;")
        lay.addWidget(lbl)

        kart = QFrame(); kart.setStyleSheet(card_ss(C['accent'] + "44")); shadow(kart)
        klay = QVBoxLayout(kart); klay.setContentsMargins(22, 18, 22, 18); klay.setSpacing(12)
        klay.addWidget(self._section_lbl("➕  Yeni Araç Ekle", C['accent']))

        grid = QGridLayout(); grid.setSpacing(10)
        self.inp_id     = QSpinBox(); self.inp_id.setRange(1, 99999); self.inp_id.setEnabled(False)
        self.inp_marka  = QComboBox(); self.inp_marka.addItems(MARKALAR)
        self.inp_model  = QLineEdit(); self.inp_model.setPlaceholderText("örn. Corolla, X5")
        self.inp_tip    = QComboBox(); self.inp_tip.addItems(ARAC_TIPLERI)
        self.inp_yakit  = QComboBox(); self.inp_yakit.addItems(YAKIT_TIPLERI)
        self.inp_sehir  = QComboBox(); self.inp_sehir.addItems(SEHIRLER)
        self.inp_km     = QSpinBox();       self.inp_km.setRange(0, 999999); self.inp_km.setSuffix(" km")


### GUI Devamı (Bölüm 2)
Arayüz kodlarının devamı...


In [ ]:
        self.inp_ucret  = QDoubleSpinBox(); self.inp_ucret.setRange(1, 9999); self.inp_ucret.setDecimals(2); self.inp_ucret.setSuffix(" ₺/km"); self.inp_ucret.setValue(3.5)

        for w in [self.inp_id, self.inp_marka, self.inp_model, self.inp_tip, self.inp_yakit, self.inp_sehir, self.inp_km, self.inp_ucret]:
            w.setStyleSheet(input_ss())

        row0 = [("Araç ID", self.inp_id), ("Marka", self.inp_marka), ("Model", self.inp_model), ("Tip", self.inp_tip)]
        row1 = [("Yakıt", self.inp_yakit), ("Şehir", self.inp_sehir), ("Kilometre", self.inp_km), ("KM Ücreti", self.inp_ucret)]
        for col, (lt, wid) in enumerate(row0):
            grid.addWidget(self._lbl(lt), 0, col); grid.addWidget(wid, 1, col)
        for col, (lt, wid) in enumerate(row1):
            grid.addWidget(self._lbl(lt), 2, col); grid.addWidget(wid, 3, col)
        klay.addLayout(grid)

        btn_lay = QHBoxLayout()
        self.btn_ekle = QPushButton("✚  Ekle")
        self.btn_guncelle = QPushButton("✎  Güncelle")
        self.btn_sil = QPushButton("🗑  Sil")
        self.btn_temizle = QPushButton("⟲  Temizle")

        self.btn_ekle.setStyleSheet(btn_primary_ss())
        self.btn_guncelle.setStyleSheet(btn_gold_ss())
        self.btn_sil.setStyleSheet(btn_danger_ss())
        self.btn_temizle.setStyleSheet(btn_outline_ss())

        for b in [self.btn_ekle, self.btn_guncelle, self.btn_sil, self.btn_temizle]:
            b.setCursor(Qt.PointingHandCursor)
            b.setFixedWidth(120)
            btn_lay.addWidget(b)
        btn_lay.addStretch()

        self.btn_ekle.clicked.connect(self._ekle)
        self.btn_guncelle.clicked.connect(self._guncelle)
        self.btn_sil.clicked.connect(self._sil)
        self.btn_temizle.clicked.connect(self._temizle)

        self.btn_guncelle.setEnabled(False)
        self.btn_sil.setEnabled(False)

        klay.addLayout(btn_lay)
        lay.addWidget(kart)

        arama_lay = QHBoxLayout()
        sub = QLabel("Tüm Araçlar")
        sub.setStyleSheet(f"color:{C['text_sub']};font-size:12px;font-weight:700;")
        arama_lay.addWidget(sub)
        arama_lay.addStretch()

        arama_lbl = QLabel("🔍  Ara:")
        arama_lbl.setStyleSheet(f"color:{C['accent']};font-size:13px;font-weight:700;")
        self.inp_arama = QLineEdit()
        self.inp_arama.setPlaceholderText("Araç ID veya Adı ile ara...")
        self.inp_arama.setStyleSheet(input_ss())
        self.inp_arama.setFixedWidth(250)
        self.inp_arama.textChanged.connect(self._arama_yap)
        arama_lay.addWidget(arama_lbl)
        arama_lay.addWidget(self.inp_arama)

        lay.addLayout(arama_lay)

        self.tablo = QTableWidget()
        self.tablo.itemSelectionChanged.connect(self._secim_degisti)
        self.tablo.setColumnCount(9)
        self.tablo.setHorizontalHeaderLabels(["ID", "Marka", "Model", "Tip", "Yakıt", "Şehir", "Kilometre", "KM Ücreti", "Durum"])
        self.tablo.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.tablo.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.tablo.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.tablo.verticalHeader().setVisible(False)
        self.tablo.setAlternatingRowColors(True)
        self.tablo.setStyleSheet(TABLE_SS)
        lay.addWidget(self.tablo)

        scroll.setWidget(container)
        main = QVBoxLayout(self); main.setContentsMargins(0, 0, 0, 0); main.addWidget(scroll)
        self._tablo_yenile()
        self._temizle()

    def _tablo_yenile(self):
        self.tablo.setRowCount(0)
        for a in self.sistem.get_araclar().values():
            r = self.tablo.rowCount(); self.tablo.insertRow(r)
            self.tablo.setItem(r, 0, QTableWidgetItem(str(a.get_arac_id())))
            self.tablo.setItem(r, 1, QTableWidgetItem(a.get_marka()))
            self.tablo.setItem(r, 2, QTableWidgetItem(a.get_model()))
            self.tablo.setItem(r, 3, QTableWidgetItem(a.get_tip()))
            self.tablo.setItem(r, 4, QTableWidgetItem(a.get_yakit_tipi()))
            self.tablo.setItem(r, 5, QTableWidgetItem(a.get_sehir()))
            self.tablo.setItem(r, 6, QTableWidgetItem(f"{a.get_kilometre():,} km"))
            self.tablo.setItem(r, 7, QTableWidgetItem(f"₺{a.get_km_ucreti():.2f}/km"))

            if a.get_musait_mi():
                d = QTableWidgetItem("✓  Müsait"); d.setForeground(QColor(C['success']))
            else:
                d = QTableWidgetItem("✗  Kirada"); d.setForeground(QColor(C['danger']))
            self.tablo.setItem(r, 8, d)
            self.tablo.setRowHeight(r, 44)

    def _bildirim(self, mesaj, ok=True):
        renk = C['success'] if ok else C['danger']
        dlg = QMessageBox(self)
        dlg.setText(mesaj)
        dlg.setWindowTitle("Sistem")
        dlg.setStyleSheet(f"QMessageBox{{background:{C['card']};}} QLabel{{color:{renk};font-size:13px;}}")
        dlg.exec_()

    def _ekle(self):
        try:
            model = self.inp_model.text().strip()
            if not model: raise ValueError("Model adı boş olamaz.")
            a = Arac(self.inp_id.value(), self.inp_marka.currentText(), model, self.inp_tip.currentText(), self.inp_yakit.currentText(), self.inp_sehir.currentText(), self.inp_km.value(), self.inp_ucret.value())
            ok, msg = self.sistem.arac_ekle(a)
            self._bildirim(msg, ok)
            if ok: 
                self._tablo_yenile()
                self._temizle()
                self.dashboard.refresh()
        except Exception as e:
            self._bildirim(str(e), False)

    def _secim_degisti(self):
        sel = self.tablo.selectedItems()
        if not sel:
            self._temizle()
            return
        r = sel[0].row()
        try:
            aid = int(self.tablo.item(r, 0).text())
            arac = self.sistem.get_araclar().get(aid)
            if arac:
                self.inp_id.setValue(arac.get_arac_id())
                self.inp_id.setEnabled(False)
                self.inp_marka.setCurrentText(arac.get_marka())
                self.inp_model.setText(arac.get_model())
                self.inp_tip.setCurrentText(arac.get_tip())
                self.inp_yakit.setCurrentText(arac.get_yakit_tipi())
                self.inp_sehir.setCurrentText(arac.get_sehir())
                self.inp_km.setValue(arac.get_kilometre())
                self.inp_ucret.setValue(arac.get_km_ucreti())

                self.btn_ekle.setEnabled(False)
                self.btn_guncelle.setEnabled(True)
                self.btn_sil.setEnabled(True)
        except Exception:
            pass

    def _temizle(self):
        araclar = self.sistem.get_araclar()
        next_id = max(araclar.keys()) + 1 if araclar else 1
        self.inp_id.setEnabled(False)
        self.inp_id.setValue(next_id)
        
        self.inp_marka.setCurrentIndex(0)
        self.inp_model.clear()
        self.inp_tip.setCurrentIndex(0)
        self.inp_yakit.setCurrentIndex(0)
        self.inp_sehir.setCurrentIndex(0)
        self.inp_km.setValue(0)
        self.inp_ucret.setValue(3.5)

        self.btn_ekle.setEnabled(True)
        self.btn_guncelle.setEnabled(False)
        self.btn_sil.setEnabled(False)
        self.tablo.clearSelection()

    def _guncelle(self):
        try:
            model = self.inp_model.text().strip()
            if not model: raise ValueError("Model adı boş olamaz.")
            ok, msg = self.sistem.arac_guncelle(
                self.inp_id.value(),
                self.inp_marka.currentText(),
                model,
                self.inp_tip.currentText(),
                self.inp_yakit.currentText(),
                self.inp_sehir.currentText(),
                self.inp_km.value(),
                self.inp_ucret.value()
            )
            self._bildirim(msg, ok)
            if ok:
                self._tablo_yenile()
                self._temizle()
                self.dashboard.refresh()
        except Exception as e:
            self._bildirim(str(e), False)

    def _sil(self):
        aid = self.inp_id.value()
        ok, msg = self.sistem.arac_sil(aid)
        self._bildirim(msg, ok)
        if ok:
            self._tablo_yenile()
            self._temizle()
            self.dashboard.refresh()

    def _arama_yap(self):
        txt = self.inp_arama.text().lower()
        for r in range(self.tablo.rowCount()):
            id_txt = self.tablo.item(r, 0).text().lower()
            marka_txt = self.tablo.item(r, 1).text().lower()
            model_txt = self.tablo.item(r, 2).text().lower()

            if txt in id_txt or txt in marka_txt or txt in model_txt:
                self.tablo.setRowHidden(r, False)
            else:
                self.tablo.setRowHidden(r, True)

# ══════════════════════════════════════════════════════════
#  EKRAN 3: Kullanıcı Yönetimi
# ══════════════════════════════════════════════════════════
class KullaniciPage(QWidget):
    def __init__(self, sistem: PaylasimSistemi, dashboard: DashboardPage, parent=None):
        super().__init__(parent)
        self.sistem    = sistem
        self.dashboard = dashboard
        self.setStyleSheet(f"background:{C['bg']};")
        self._build()

    def _lbl(self, txt):
        l = QLabel(txt)
        l.setStyleSheet(f"color:{C['text_sub']};font-size:11px;font-weight:700;")
        return l

    def _build(self):
        lay = QVBoxLayout(self)
        lay.setContentsMargins(28, 22, 28, 22)
        lay.setSpacing(16)

        lbl = QLabel("👤  Kullanıcı Yönetimi")
        lbl.setStyleSheet(f"color:{C['text']};font-size:22px;font-weight:800;")
        lay.addWidget(lbl)

        kart = QFrame(); kart.setStyleSheet(card_ss(C['gold'] + "66")); shadow(kart)
        klay = QVBoxLayout(kart); klay.setContentsMargins(22, 18, 22, 18); klay.setSpacing(12)
        sl = QLabel("➕  Yeni Kullanıcı Kaydı")
        sl.setStyleSheet(f"color:{C['gold']};font-size:13px;font-weight:700;background:transparent;")
        klay.addWidget(sl)

        grid = QGridLayout(); grid.setSpacing(10)
        self.inp_kid    = QSpinBox(); self.inp_kid.setRange(1, 99999); self.inp_kid.setEnabled(False)
        self.inp_ad     = QLineEdit(); self.inp_ad.setPlaceholderText("Ad Soyad")
        self.inp_ad.setValidator(QRegExpValidator(QRegExp(r"^[A-Za-zÇçĞğİıÖöŞşÜü\s]+$"), self))
        self.inp_ehliyet= QLineEdit(); self.inp_ehliyet.setPlaceholderText("Örn: 34ABC123")
        self.inp_ehliyet.setValidator(QRegExpValidator(QRegExp(r"^\d{2}[A-Za-z]{3}\d{3}$"), self))
        self.inp_tel    = QLineEdit(); self.inp_tel.setPlaceholderText("Telefon")
        self.inp_tel.setMaxLength(14)
        self.inp_tel.textChanged.connect(self._format_tel)

        custom_ss = input_ss() + """
            QLineEdit, QSpinBox {
                border: 1.5px solid #D1D5DB;
                background: #FFFFFF;
            }
            QLineEdit:focus, QSpinBox:focus {
                border: 1.5px solid #9CA3AF;
            }
        """
        for w in [self.inp_kid, self.inp_ad, self.inp_ehliyet, self.inp_tel]:
            w.setStyleSheet(custom_ss)

        cols = [("Kullanıcı ID", self.inp_kid), ("Ad Soyad", self.inp_ad), ("Ehliyet No", self.inp_ehliyet), ("Telefon", self.inp_tel)]
        for col, (lt, wid) in enumerate(cols):
            grid.addWidget(self._lbl(lt), 0, col); grid.addWidget(wid, 1, col)
        klay.addLayout(grid)

        btn_lay = QHBoxLayout()
        btn = QPushButton("✚  Kullanıcı Ekle")
        btn.setStyleSheet(btn_gold_ss()); btn.setCursor(Qt.PointingHandCursor)
        btn.clicked.connect(self._ekle); btn.setFixedWidth(170)
        btn_temizle = QPushButton("⟲  Temizle")
        btn_temizle.setStyleSheet(btn_outline_ss()); btn_temizle.setCursor(Qt.PointingHandCursor)
        btn_temizle.clicked.connect(self._temizle); btn_temizle.setFixedWidth(120)
        btn_lay.addWidget(btn)
        btn_lay.addWidget(btn_temizle)
        btn_lay.addStretch()
        klay.addLayout(btn_lay)
        lay.addWidget(kart)

        kart2 = QFrame(); kart2.setStyleSheet(card_ss()); shadow(kart2)
        k2lay = QVBoxLayout(kart2); k2lay.setContentsMargins(22, 16, 22, 16); k2lay.setSpacing(10)
        sl2 = QLabel("📋  Kullanıcı Kiralama Geçmişi")
        sl2.setStyleSheet(f"color:{C['accent']};font-size:13px;font-weight:700;background:transparent;")
        k2lay.addWidget(sl2)
        row = QHBoxLayout(); row.setSpacing(10)
        self.g_kid = QSpinBox(); self.g_kid.setRange(1, 9999)
        self.g_kid.setStyleSheet(input_ss()); self.g_kid.setFixedWidth(120)
        row.addWidget(self._lbl("Kullanıcı ID")); row.addWidget(self.g_kid)
        b_sor = QPushButton("🔍  Sorgula"); b_sor.setStyleSheet(btn_outline_ss())
        b_sor.setCursor(Qt.PointingHandCursor); b_sor.clicked.connect(self._sorgula)
        row.addWidget(b_sor); row.addStretch()
        k2lay.addLayout(row)
        lay.addWidget(kart2)

        arama_lay = QHBoxLayout()
        sub = QLabel("Kayıtlı Kullanıcılar")
        sub.setStyleSheet(f"color:{C['text_sub']};font-size:12px;font-weight:700;")
        arama_lay.addWidget(sub)
        arama_lay.addStretch()
        
        arama_lbl = QLabel("🔍  Ara:")
        arama_lbl.setStyleSheet(f"color:{C['accent']};font-size:13px;font-weight:700;")
        self.inp_arama = QLineEdit()
        self.inp_arama.setPlaceholderText("Kullanıcı ID veya Adı ile ara...")
        self.inp_arama.setStyleSheet(input_ss())
        self.inp_arama.setFixedWidth(250)
        self.inp_arama.textChanged.connect(self._arama_yap)
        arama_lay.addWidget(arama_lbl)
        arama_lay.addWidget(self.inp_arama)
        
        lay.addLayout(arama_lay)

        self.tablo = QTableWidget()
        self.tablo.setColumnCount(5)
        self.tablo.setHorizontalHeaderLabels(["ID", "Ad Soyad", "Ehliyet No", "Telefon", "Durum"])
        self.tablo.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.tablo.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.tablo.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.tablo.verticalHeader().setVisible(False)
        self.tablo.setAlternatingRowColors(True)
        self.tablo.setStyleSheet(TABLE_SS)
        lay.addWidget(self.tablo)
        self._tablo_yenile()
        self._temizle()

    def _format_tel(self, text):
        if self.inp_tel.signalsBlocked(): return
        digits = "".join(filter(str.isdigit, text))
        f = ""
        if len(digits) > 0: f += digits[:4]
        if len(digits) > 4: f += " " + digits[4:7]
        if len(digits) > 7: f += " " + digits[7:9]
        if len(digits) > 9: f += " " + digits[9:11]
        
        self.inp_tel.blockSignals(True)
        self.inp_tel.setText(f)
        self.inp_tel.setCursorPosition(len(f))
        self.inp_tel.blockSignals(False)

    def _temizle(self):
        kullanicilar = self.sistem.get_kullanicilar()
        next_id = max(kullanicilar.keys()) + 1 if kullanicilar else 1
        self.inp_kid.setValue(next_id)
        self.inp_ad.clear()
        self.inp_ehliyet.clear()
        self.inp_tel.clear()
        self.tablo.clearSelection()

    def _arama_yap(self):
        txt = self.inp_arama.text().lower()
        for r in range(self.tablo.rowCount()):
            id_txt = self.tablo.item(r, 0).text().lower()


### GUI Devamı (Bölüm 3)
Arayüz kodlarının devamı...


In [ ]:
            ad_txt = self.tablo.item(r, 1).text().lower()
            if txt in id_txt or txt in ad_txt:
                self.tablo.setRowHidden(r, False)
            else:
                self.tablo.setRowHidden(r, True)

    def _tablo_yenile(self):
        self.tablo.setRowCount(0)
        for u in self.sistem.get_kullanicilar().values():
            r = self.tablo.rowCount(); self.tablo.insertRow(r)
            self.tablo.setItem(r, 0, QTableWidgetItem(str(u.get_kullanici_id())))
            self.tablo.setItem(r, 1, QTableWidgetItem(u.get_ad()))
            self.tablo.setItem(r, 2, QTableWidgetItem(u.get_ehliyet_no()))
            self.tablo.setItem(r, 3, QTableWidgetItem(u.get_telefon() or "—"))
            aktif = u.aktif_kiralama()
            if aktif:
                d = QTableWidgetItem(f"🔑  Araçta ({aktif.get_arac().get_tam_ad()})")
                d.setForeground(QColor(C['warning']))
            else:
                d = QTableWidgetItem("✓  Müsait"); d.setForeground(QColor(C['success']))
            self.tablo.setItem(r, 4, d)
            self.tablo.setRowHeight(r, 44)

    def _bildirim(self, mesaj, ok=True):
        renk = C['success'] if ok else C['danger']
        dlg = QMessageBox(self)
        dlg.setText(mesaj)
        dlg.setWindowTitle("Sistem")
        dlg.setStyleSheet(f"QMessageBox{{background:{C['card']};}} QLabel{{color:{renk};font-size:13px;}}")
        dlg.exec_()

    def _ekle(self):
        try:
            ad = self.inp_ad.text().strip()
            if not ad: raise ValueError("Ad boş olamaz.")
            ehliyet = self.inp_ehliyet.text().strip()
            if not ehliyet: raise ValueError("Ehliyet numarası zorunludur.")
            u = Kullanici(self.inp_kid.value(), ad, ehliyet, self.inp_tel.text().strip())
            ok, msg = self.sistem.kullanici_ekle(u)
            self._bildirim(msg, ok)
            if ok: 
                self._tablo_yenile()
                self._temizle()
                self.dashboard.refresh()
        except Exception as e:
            self._bildirim(str(e), False)

    def _sorgula(self):
        uid = self.g_kid.value()
        kullanicilar = self.sistem.get_kullanicilar()
        if uid not in kullanicilar:
            self._bildirim("Kullanıcı bulunamadı.", False); return
        u     = kullanicilar[uid]
        gecmis = u.kiralama_gecmisi()
        if not gecmis:
            self._bildirim(f"'{u.get_ad()}' için kiralama geçmişi yok."); return

        mesaj_satırlar = [f"📋  {u.get_ad()} — Kiralama Geçmişi\n"]
        for k in gecmis:
            b = k.kiralama_bilgisi()
            mesaj_satırlar.append(f"#{b['id']}  {b['arac']}  |  {b['baslangic']} → {b['bitis']}  |  Süre: {b['sure']}  |  Ücret: {b['ucret']}")
        self._bildirim("\n".join(mesaj_satırlar))

# ══════════════════════════════════════════════════════════
#  EKRAN 4: Kiralama İşlemleri
# ══════════════════════════════════════════════════════════
class KiralamePage(QWidget):
    def __init__(self, sistem: PaylasimSistemi, dashboard: DashboardPage, parent=None):
        super().__init__(parent)
        self.sistem    = sistem
        self.dashboard = dashboard
        self.setStyleSheet(f"background:{C['bg']};")
        self._build()

    def _lbl(self, txt):
        l = QLabel(txt)
        l.setStyleSheet(f"color:{C['text_sub']};font-size:11px;font-weight:700;")
        return l

    def _build(self):
        scroll = QScrollArea(self)
        scroll.setWidgetResizable(True)
        scroll.setStyleSheet("QScrollArea{border:none;background:transparent;}")
        container = QWidget(); container.setStyleSheet(f"background:{C['bg']};")
        lay = QVBoxLayout(container)
        lay.setContentsMargins(28, 22, 28, 22)
        lay.setSpacing(16)

        lbl = QLabel("🔑  Kiralama İşlemleri")
        lbl.setStyleSheet(f"color:{C['text']};font-size:22px;font-weight:800;")
        lay.addWidget(lbl)

        top_row = QHBoxLayout()
        top_row.setSpacing(16)

        # Başlat Kartı
        kart = QFrame(); kart.setStyleSheet(card_ss(C['success'] + "66")); shadow(kart)
        klay = QVBoxLayout(kart); klay.setContentsMargins(22, 18, 22, 18); klay.setSpacing(12)
        sl = QLabel("▶  Kiralama Başlat")
        sl.setStyleSheet(f"color:{C['success']};font-size:13px;font-weight:700;background:transparent;")
        klay.addWidget(sl)
        grid = QGridLayout(); grid.setSpacing(10)
        self.b_arac_id = QSpinBox(); self.b_arac_id.setRange(1, 99999)
        self.b_user_id = QSpinBox(); self.b_user_id.setRange(1, 99999)
        self.b_alis_sehri = QComboBox(); self.b_alis_sehri.addItems(SEHIRLER)
        self.b_birakis_sehri = QComboBox(); self.b_birakis_sehri.addItems(SEHIRLER)
        self.b_bas_dt  = QDateTimeEdit(); self.b_bas_dt.setCalendarPopup(True)
        self.b_bas_dt.setDateTime(QDateTime.currentDateTime())

        for w in [self.b_arac_id, self.b_user_id, self.b_alis_sehri, self.b_birakis_sehri, self.b_bas_dt]:
            w.setStyleSheet(input_ss())

        cols = [("Araç ID", self.b_arac_id), ("Kullanıcı ID", self.b_user_id), ("Alış Şehri", self.b_alis_sehri)]
        cols_b = [("Bırakış Şehri", self.b_birakis_sehri), ("Başlangıç", self.b_bas_dt)]
        for col, (lt, wid) in enumerate(cols):
            grid.addWidget(self._lbl(lt), 0, col); grid.addWidget(wid, 1, col)
        for col, (lt, wid) in enumerate(cols_b):
            grid.addWidget(self._lbl(lt), 2, col); grid.addWidget(wid, 3, col)
        
        klay.addLayout(grid)
        btn_bas = QPushButton("▶  Kiralama Başlat"); btn_bas.setStyleSheet(btn_success_ss())
        btn_bas.setCursor(Qt.PointingHandCursor); btn_bas.clicked.connect(self._baslat)
        btn_bas.setFixedWidth(170)
        klay.addWidget(btn_bas)
        top_row.addWidget(kart)

        # Bitir Kartı
        kart2 = QFrame(); kart2.setStyleSheet(card_ss(C['danger'] + "55")); shadow(kart2)
        k2lay = QVBoxLayout(kart2); k2lay.setContentsMargins(22, 18, 22, 18); k2lay.setSpacing(12)
        sl2 = QLabel("■  Kiralama Bitir")
        sl2.setStyleSheet(f"color:{C['danger']};font-size:13px;font-weight:700;background:transparent;")
        k2lay.addWidget(sl2)
        grid2 = QGridLayout(); grid2.setSpacing(10)
        self.e_kid  = QSpinBox(); self.e_kid.setRange(1, 99999)
        self.e_bit_dt = QDateTimeEdit(); self.e_bit_dt.setCalendarPopup(True)
        self.e_bit_dt.setDateTime(QDateTime.currentDateTime())
        self.e_ucret = QLineEdit(); self.e_ucret.setReadOnly(True)
        self.e_ucret.setPlaceholderText("Seçim Bekleniyor...")
        for w in [self.e_kid, self.e_bit_dt, self.e_ucret]: w.setStyleSheet(input_ss())
        cols2 = [("Kiralama ID", self.e_kid), ("Bitiş Saati", self.e_bit_dt)]
        for col, (lt, wid) in enumerate(cols2):
            grid2.addWidget(self._lbl(lt), 0, col); grid2.addWidget(wid, 1, col)
        
        grid2.addWidget(self._lbl("Hesaplanan Ücret"), 2, 0)
        grid2.addWidget(self.e_ucret, 3, 0, 1, 3)
        k2lay.addLayout(grid2)
        
        self.e_kid.valueChanged.connect(self._hesapla_bitis_ucret)
        self.e_bit_dt.dateTimeChanged.connect(self._hesapla_bitis_ucret)
        
        k2lay.addStretch()
        btn_bit = QPushButton("■  Kiralama Bitir"); btn_bit.setStyleSheet(btn_danger_ss())
        btn_bit.setCursor(Qt.PointingHandCursor); btn_bit.clicked.connect(self._bitir)
        btn_bit.setFixedWidth(170)
        k2lay.addWidget(btn_bit)
        top_row.addWidget(kart2)

        lay.addLayout(top_row)

        # Selection Tables Area
        mid_row = QHBoxLayout()
        mid_row.setSpacing(16)
        
        arac_lay = QVBoxLayout()
        arac_ara_lbl = QLabel("🚗  Araç Seç")
        arac_ara_lbl.setStyleSheet(f"color:{C['text_sub']};font-size:12px;font-weight:700;")
        self.inp_arac_ara = QLineEdit()
        self.inp_arac_ara.setPlaceholderText("Araç ID veya Adı ile ara...")
        self.inp_arac_ara.setStyleSheet(input_ss())
        self.inp_arac_ara.textChanged.connect(self._arac_ara_yap)
        
        self.arac_tablo = QTableWidget()
        self.arac_tablo.setColumnCount(2)
        self.arac_tablo.setHorizontalHeaderLabels(["ID", "Araç"])
        self.arac_tablo.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.arac_tablo.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.arac_tablo.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.arac_tablo.verticalHeader().setVisible(False)
        self.arac_tablo.setAlternatingRowColors(True)
        self.arac_tablo.setStyleSheet(TABLE_SS)
        self.arac_tablo.setFixedHeight(150)
        self.arac_tablo.itemSelectionChanged.connect(self._arac_secildi)
        
        arac_lay.addWidget(arac_ara_lbl)
        arac_lay.addWidget(self.inp_arac_ara)
        arac_lay.addWidget(self.arac_tablo)
        
        kul_lay = QVBoxLayout()
        kul_ara_lbl = QLabel("👤  Kullanıcı Seç")
        kul_ara_lbl.setStyleSheet(f"color:{C['text_sub']};font-size:12px;font-weight:700;")
        self.inp_kul_ara = QLineEdit()
        self.inp_kul_ara.setPlaceholderText("Kullanıcı ID veya Adı ile ara...")
        self.inp_kul_ara.setStyleSheet(input_ss())
        self.inp_kul_ara.textChanged.connect(self._kul_ara_yap)
        
        self.kul_tablo = QTableWidget()
        self.kul_tablo.setColumnCount(2)
        self.kul_tablo.setHorizontalHeaderLabels(["ID", "Ad Soyad"])
        self.kul_tablo.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.kul_tablo.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.kul_tablo.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.kul_tablo.verticalHeader().setVisible(False)
        self.kul_tablo.setAlternatingRowColors(True)
        self.kul_tablo.setStyleSheet(TABLE_SS)
        self.kul_tablo.setFixedHeight(150)
        self.kul_tablo.itemSelectionChanged.connect(self._kul_secildi)
        
        kul_lay.addWidget(kul_ara_lbl)
        kul_lay.addWidget(self.inp_kul_ara)
        kul_lay.addWidget(self.kul_tablo)

        mid_row.addLayout(arac_lay)
        mid_row.addLayout(kul_lay)
        lay.addLayout(mid_row)

        bot_lay = QVBoxLayout()
        bot_header = QHBoxLayout()
        sub = QLabel("Tüm Kiralamalar")
        sub.setStyleSheet(f"color:{C['text_sub']};font-size:12px;font-weight:700;")
        
        genel_ara_lbl = QLabel("🔍  Genel Sorgulama:")
        genel_ara_lbl.setStyleSheet(f"color:{C['accent']};font-size:12px;font-weight:700;")
        self.inp_genel_ara = QLineEdit()
        self.inp_genel_ara.setPlaceholderText("ID, Araç veya Kullanıcı Ara...")
        self.inp_genel_ara.setStyleSheet(input_ss())
        self.inp_genel_ara.setFixedWidth(250)
        self.inp_genel_ara.textChanged.connect(self._genel_ara_yap)
        
        bot_header.addWidget(sub)
        bot_header.addStretch()
        bot_header.addWidget(genel_ara_lbl)
        bot_header.addWidget(self.inp_genel_ara)
        bot_lay.addLayout(bot_header)

        self.tablo = QTableWidget()
        self.tablo.setColumnCount(8)
        self.tablo.setHorizontalHeaderLabels(["ID", "Araç", "Kullanıcı", "Alış Şehri", "Bırakış Şehri", "Başlangıç", "Bitiş", "Ücret"])
        self.tablo.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.tablo.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.tablo.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.tablo.verticalHeader().setVisible(False)
        self.tablo.setAlternatingRowColors(True)
        self.tablo.setStyleSheet(TABLE_SS)
        self.tablo.itemSelectionChanged.connect(self._kiralama_secildi)
        bot_lay.addWidget(self.tablo)
        lay.addLayout(bot_lay)

        scroll.setWidget(container)
        main = QVBoxLayout(self); main.setContentsMargins(0, 0, 0, 0); main.addWidget(scroll)
        self._tablo_yenile()

    def _hesapla_bitis_ucret(self):
        kid = self.e_kid.value()
        kiralama = next((k for k in self.sistem.get_aktif_kiralamalar() if k.get_kiralama_id() == kid), None)
        if not kiralama:
            self.e_ucret.setText("Kiralama Bulunamadı")
            return
        
        mesafe = get_mesafe(kiralama.get_alis_sehri(), kiralama.get_birakis_sehri())
        toplam = mesafe * kiralama.get_arac().get_km_ucreti()
        
        self.e_ucret.setText(f"₺{toplam:,.2f} ({mesafe} km)")

    def _arac_secildi(self):
        sel = self.arac_tablo.selectedItems()
        if sel:
            self.b_arac_id.setValue(int(sel[0].text()))

    def _kul_secildi(self):
        sel = self.kul_tablo.selectedItems()
        if sel:
            self.b_user_id.setValue(int(sel[0].text()))

    def _kiralama_secildi(self):
        sel = self.tablo.selectedItems()
        if sel:
            self.e_kid.setValue(int(sel[0].text()))
            self.e_bit_dt.setDateTime(QDateTime.currentDateTime())

    def _arac_ara_yap(self):
        txt = self.inp_arac_ara.text().lower()
        for r in range(self.arac_tablo.rowCount()):
            id_txt = self.arac_tablo.item(r, 0).text().lower()
            ad_txt = self.arac_tablo.item(r, 1).text().lower()
            if txt in id_txt or txt in ad_txt:
                self.arac_tablo.setRowHidden(r, False)
            else:
                self.arac_tablo.setRowHidden(r, True)

    def _kul_ara_yap(self):
        txt = self.inp_kul_ara.text().lower()
        for r in range(self.kul_tablo.rowCount()):
            id_txt = self.kul_tablo.item(r, 0).text().lower()
            ad_txt = self.kul_tablo.item(r, 1).text().lower()
            if txt in id_txt or txt in ad_txt:
                self.kul_tablo.setRowHidden(r, False)
            else:
                self.kul_tablo.setRowHidden(r, True)

    def _genel_ara_yap(self):
        txt = self.inp_genel_ara.text().lower()
        for r in range(self.tablo.rowCount()):
            id_txt = self.tablo.item(r, 0).text().lower()
            arac_txt = self.tablo.item(r, 1).text().lower()
            kul_txt = self.tablo.item(r, 2).text().lower()
            if txt in id_txt or txt in arac_txt or txt in kul_txt:
                self.tablo.setRowHidden(r, False)
            else:
                self.tablo.setRowHidden(r, True)

    def _tablo_yenile(self):
        self.tablo.setRowCount(0)
        for k in reversed(self.sistem.get_aktif_kiralamalar()):
            b = k.kiralama_bilgisi()
            r = self.tablo.rowCount(); self.tablo.insertRow(r)
            self.tablo.setItem(r, 0, QTableWidgetItem(str(b["id"])))
            self.tablo.setItem(r, 1, QTableWidgetItem(b["arac"]))
            self.tablo.setItem(r, 2, QTableWidgetItem(b["kullanici"]))
            self.tablo.setItem(r, 3, QTableWidgetItem(b["alis"]))
            self.tablo.setItem(r, 4, QTableWidgetItem(b["birakis"]))
            self.tablo.setItem(r, 5, QTableWidgetItem(b["baslangic"]))
            self.tablo.setItem(r, 6, QTableWidgetItem(b["bitis"]))
            ucret_item = QTableWidgetItem(b["ucret"])
            if b["aktif"]:
                ucret_item.setForeground(QColor(C['warning']))
            else:
                ucret_item.setForeground(QColor(C['success']))
            self.tablo.setItem(r, 7, ucret_item)
            self.tablo.setRowHeight(r, 44)

        self.arac_tablo.setRowCount(0)
        for a in self.sistem.get_araclar().values():
            if not a.get_musait_mi(): continue
            r = self.arac_tablo.rowCount(); self.arac_tablo.insertRow(r)
            self.arac_tablo.setItem(r, 0, QTableWidgetItem(str(a.get_arac_id())))
            self.arac_tablo.setItem(r, 1, QTableWidgetItem(a.get_tam_ad()))
            self.arac_tablo.setRowHeight(r, 30)

        self.kul_tablo.setRowCount(0)
        for u in self.sistem.get_kullanicilar().values():
            if u.aktif_kiralama(): continue
            r = self.kul_tablo.rowCount(); self.kul_tablo.insertRow(r)
            self.kul_tablo.setItem(r, 0, QTableWidgetItem(str(u.get_kullanici_id())))
            self.kul_tablo.setItem(r, 1, QTableWidgetItem(u.get_ad()))
            self.kul_tablo.setRowHeight(r, 30)

    def _bildirim(self, mesaj, ok=True):
        renk = C['success'] if ok else C['danger']
        dlg = QMessageBox(self)
        dlg.setText(mesaj)


### GUI Devamı (Bölüm 4)
Arayüz kodlarının devamı...


In [ ]:
        dlg.setWindowTitle("Kiralama")
        dlg.setStyleSheet(f"QMessageBox{{background:{C['card']};}} QLabel{{color:{renk};font-size:13px;}}")
        dlg.exec_()

    def _qdt_to_dt(self, qdt):
        return datetime(qdt.date().year(), qdt.date().month(), qdt.date().day(),
                        qdt.time().hour(), qdt.time().minute())

    def _baslat(self):
        baslangic = self._qdt_to_dt(self.b_bas_dt.dateTime())
        ok, msg   = self.sistem.kiralama_baslat(
            self.b_arac_id.value(), self.b_user_id.value(), self.b_alis_sehri.currentText(), self.b_birakis_sehri.currentText(), baslangic)
        self._bildirim(msg, ok)
        if ok: self._tablo_yenile(); self.dashboard.refresh()

    def _bitir(self):
        bitis  = self._qdt_to_dt(self.e_bit_dt.dateTime())
        ok, msg = self.sistem.kiralama_bitir(
            self.e_kid.value(), bitis)
        self._bildirim(msg, ok)
        if ok: self._tablo_yenile(); self.dashboard.refresh()

# ══════════════════════════════════════════════════════════
#  EKRAN 5: Gelir Raporu
# ══════════════════════════════════════════════════════════
class RaporPage(QWidget):
    def __init__(self, sistem: PaylasimSistemi, parent=None):
        super().__init__(parent)
        self.sistem = sistem
        self.setStyleSheet(f"background:{C['bg']};")
        self._build()

    def _build(self):
        lay = QVBoxLayout(self)
        lay.setContentsMargins(28, 22, 28, 22)
        lay.setSpacing(16)

        ust = QHBoxLayout()
        lbl = QLabel("📊  Gelir & Performans Raporu")
        lbl.setStyleSheet(f"color:{C['text']};font-size:22px;font-weight:800;")
        ust.addWidget(lbl); ust.addStretch()
        btn = QPushButton("↻  Yenile"); btn.setStyleSheet(btn_outline_ss())
        btn.setCursor(Qt.PointingHandCursor); btn.clicked.connect(self.refresh)
        ust.addWidget(btn)
        lay.addLayout(ust)

        self.kpi_row = QHBoxLayout(); self.kpi_row.setSpacing(14)
        lay.addLayout(self.kpi_row)

        bot_header = QHBoxLayout()
        sub = QLabel("Tamamlanan Kiralamalar — Detay")
        sub.setStyleSheet(f"color:{C['text_sub']};font-size:12px;font-weight:700;")
        bot_header.addWidget(sub)
        bot_header.addStretch()

        genel_ara_lbl = QLabel("🔍  Ara:")
        genel_ara_lbl.setStyleSheet(f"color:{C['accent']};font-size:12px;font-weight:700;")
        self.inp_rapor_ara = QLineEdit()
        self.inp_rapor_ara.setPlaceholderText("Araç veya Kullanıcı Ara...")
        self.inp_rapor_ara.setStyleSheet(input_ss())
        self.inp_rapor_ara.setFixedWidth(250)
        self.inp_rapor_ara.textChanged.connect(self._arama_yap)
        bot_header.addWidget(genel_ara_lbl)
        bot_header.addWidget(self.inp_rapor_ara)

        lay.addLayout(bot_header)

        self.tablo = QTableWidget()
        self.tablo.setColumnCount(6)
        self.tablo.setHorizontalHeaderLabels(["Araç", "Kullanıcı", "Başlangıç", "Bitiş", "Süre", "Ücret"])
        self.tablo.horizontalHeader().setSectionResizeMode(QHeaderView.Stretch)
        self.tablo.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.tablo.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.tablo.verticalHeader().setVisible(False)
        self.tablo.setAlternatingRowColors(True)
        self.tablo.setStyleSheet(TABLE_SS)
        lay.addWidget(self.tablo)

        self.lbl_toplam = QLabel()
        self.lbl_toplam.setAlignment(Qt.AlignRight)
        self.lbl_toplam.setStyleSheet(f"""
            color:{C['accent']};font-size:15px;font-weight:800;
            padding:12px 18px;background:{C['card']};
            border:1.5px solid {C['border']};border-radius:10px;
        """)
        lay.addWidget(self.lbl_toplam)
        self.refresh()

    def refresh(self):
        gecmis = self.sistem.get_gecmis_kiralamalar()
        aktif_k = len(self.sistem.get_aktif_kiralamalar())
        toplam_gelir = self.sistem.toplam_gelir()
        musait = len(self.sistem.get_musait_araclar())

        while self.kpi_row.count():
            item = self.kpi_row.takeAt(0)
            if item.widget(): item.widget().deleteLater()

        self.kpi_row.addWidget(KpiCard("💰", "Toplam Gelir",       f"₺{toplam_gelir:,.2f}", "tüm zamanlar",         C['success']))
        self.kpi_row.addWidget(KpiCard("📋", "Tamamlanan",         len(gecmis),             "kiralama işlemi",       C['accent']))
        self.kpi_row.addWidget(KpiCard("🔑", "Devam Eden",         aktif_k,                 "aktif kiralama",        C['warning']))
        self.kpi_row.addWidget(KpiCard("🚗", "Müsait Araç",        musait,                  "şu an kiralanabilir",   C['gold']))

        self.tablo.setRowCount(0)
        for k in reversed(gecmis):
            b = k.kiralama_bilgisi()
            r = self.tablo.rowCount(); self.tablo.insertRow(r)
            self.tablo.setItem(r, 0, QTableWidgetItem(b["arac"]))
            self.tablo.setItem(r, 1, QTableWidgetItem(b["kullanici"]))
            self.tablo.setItem(r, 2, QTableWidgetItem(b["baslangic"]))
            self.tablo.setItem(r, 3, QTableWidgetItem(b["bitis"]))
            self.tablo.setItem(r, 4, QTableWidgetItem(b["sure"]))
            ucret = QTableWidgetItem(b["ucret"]); ucret.setForeground(QColor(C['success']))
            self.tablo.setItem(r, 5, ucret)
            self.tablo.setRowHeight(r, 44)

        self.lbl_toplam.setText(f"Toplam Gelir:  ₺{toplam_gelir:,.2f}  |  İşlem Sayısı: {len(gecmis)}")

    def _arama_yap(self):
        txt = self.inp_rapor_ara.text().lower()
        for r in range(self.tablo.rowCount()):
            arac_txt = self.tablo.item(r, 0).text().lower()
            kul_txt = self.tablo.item(r, 1).text().lower()
            if txt in arac_txt or txt in kul_txt:
                self.tablo.setRowHidden(r, False)
            else:
                self.tablo.setRowHidden(r, True)

# ══════════════════════════════════════════════════════════
#  ANA PENCERE
# ══════════════════════════════════════════════════════════
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("🚗  CarShare — Araç Paylaşım Yönetim Sistemi")
        self.setMinimumSize(1200, 720)
        self.resize(1380, 820)
        self.setStyleSheet(f"QMainWindow{{background:{C['bg']};color:{C['text']};}}")
        self.sistem = PaylasimSistemi()
        self._build_ui()
        self._saat_timer()

    def _demo_veri(self):
        return PaylasimSistemi()

    def _build_ui(self):
        merkez = QWidget(); merkez.setStyleSheet(f"background:{C['bg']};")
        self.setCentralWidget(merkez)
        ana = QHBoxLayout(merkez); ana.setContentsMargins(0, 0, 0, 0); ana.setSpacing(0)

        sidebar = QFrame(); sidebar.setFixedWidth(C['sidebar_w'])
        sidebar.setStyleSheet(f"QFrame{{background:{C['sidebar']};border-right:1.5px solid {C['border']};}}")
        shadow(sidebar, blur=24, dy=0, alpha=18)
        sb = QVBoxLayout(sidebar); sb.setContentsMargins(0, 0, 0, 18); sb.setSpacing(0)

        logo = QFrame(); logo.setFixedHeight(88)
        logo.setStyleSheet(f"""
            background:qlineargradient(x1:0,y1:0,x2:1,y2:1,
                stop:0 {C['accent2']},stop:0.5 {C['accent']},stop:1 {C['accent3']});
        """)
        ll = QVBoxLayout(logo); ll.setContentsMargins(18, 14, 18, 14)
        t1 = QLabel("🚗  CarShare")
        t1.setStyleSheet("color:white;font-size:20px;font-weight:800;background:transparent;letter-spacing:-0.5px;")
        t2 = QLabel("Araç Paylaşım Sistemi")
        t2.setStyleSheet("color:rgba(255,255,255,0.75);font-size:10px;background:transparent;letter-spacing:0.3px;")
        ll.addWidget(t1); ll.addWidget(t2)
        sb.addWidget(logo); sb.addSpacing(14)

        self.nav = []
        pages = [("📊", "Dashboard"), ("🚗", "Araç Yönetimi"), ("👤", "Kullanıcılar"), ("🔑", "Kiralamalar"), ("📊", "Raporlar")]
        for ikon, metin in pages:
            btn = SideBtn(ikon, metin)
            btn.clicked.connect(lambda _, m=metin: self._goto(m))
            sb.addWidget(btn); self.nav.append(btn)

        sb.addStretch()
        self.lbl_saat = QLabel()
        self.lbl_saat.setAlignment(Qt.AlignCenter)
        self.lbl_saat.setStyleSheet(f"color:{C['text_dim']};font-size:11px;background:transparent;padding-bottom:4px;")
        sb.addWidget(self.lbl_saat)
        ana.addWidget(sidebar)

        icerik = QWidget(); icerik.setStyleSheet(f"background:{C['bg']};")
        il = QVBoxLayout(icerik); il.setContentsMargins(0, 0, 0, 0); il.setSpacing(0)

        topbar = QFrame(); topbar.setFixedHeight(52)
        topbar.setStyleSheet(f"background:{C['sidebar']};border-bottom:1.5px solid {C['border']};")
        tl = QHBoxLayout(topbar); tl.setContentsMargins(24, 0, 24, 0)
        self.lbl_page = QLabel("Dashboard")
        self.lbl_page.setStyleSheet(f"color:{C['text_sub']};font-size:12px;font-weight:600;background:transparent;")
        tl.addWidget(self.lbl_page); tl.addStretch()
        badge = QLabel("🚗  Filoya Yönetici")
        badge.setStyleSheet(f"""
            color:{C['accent']};font-size:12px;font-weight:600;
            background:{C['tag_bg']};border:1px solid {C['border']};
            border-radius:14px;padding:3px 12px;
        """)
        tl.addWidget(badge)
        il.addWidget(topbar)

        self.stack = QStackedWidget(); self.stack.setStyleSheet(f"background:{C['bg']};")
        self.p_dash    = DashboardPage(self.sistem)
        self.p_arac    = AracPage(self.sistem, self.p_dash)
        self.p_kullanici = KullaniciPage(self.sistem, self.p_dash)
        self.p_kiralama  = KiralamePage(self.sistem, self.p_dash)
        self.p_rapor     = RaporPage(self.sistem)
        for p in [self.p_dash, self.p_arac, self.p_kullanici, self.p_kiralama, self.p_rapor]:
            self.stack.addWidget(p)
        il.addWidget(self.stack)
        ana.addWidget(icerik)
        self._goto("Dashboard")

    def _goto(self, sayfa):
        idx = {"Dashboard": 0, "Araç Yönetimi": 1, "Kullanıcılar": 2, "Kiralamalar": 3, "Raporlar": 4}[sayfa]
        self.stack.setCurrentIndex(idx)
        self.lbl_page.setText(f"Ana Sayfa  ›  {sayfa}")
        for i, b in enumerate(self.nav): b.setChecked(i == idx)
        if idx == 0: self.p_dash.refresh()
        if idx == 4: self.p_rapor.refresh()

    def _saat_timer(self):
        self._saat_guncelle()
        t = QTimer(self); t.timeout.connect(self._saat_guncelle); t.start(1000)

    def _saat_guncelle(self):
        self.lbl_saat.setText(datetime.now().strftime("%d.%m.%Y\n%H:%M:%S"))

def main():
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    p = QPalette()
    p.setColor(QPalette.Window,          QColor(C['bg']))
    p.setColor(QPalette.WindowText,      QColor(C['text']))
    p.setColor(QPalette.Base,            QColor(C['card']))
    p.setColor(QPalette.AlternateBase,   QColor(C['row_alt']))
    p.setColor(QPalette.Text,            QColor(C['text']))
    p.setColor(QPalette.Button,          QColor(C['card']))
    p.setColor(QPalette.ButtonText,      QColor(C['text']))
    p.setColor(QPalette.Highlight,       QColor(C['accent']))
    p.setColor(QPalette.HighlightedText, QColor("#FFFFFF"))
    app.setPalette(p)

    win = MainWindow()
    win.show()
    sys.exit(app.exec_())

if __name__ == "__main__":
    main()